In [22]:
import os

import numpy as np
import pandas as pd
import psycopg

from dotenv import load_dotenv

load_dotenv()

True

In [23]:
connection = psycopg.connect(
    host=os.getenv("POSTGRES_HOST"),
    port=os.getenv("POSTGRES_PORT"),
    dbname=os.getenv("POSTGRES_DB"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
)

query = """
SELECT
    date,
    period,
    net_load_mwh,
    crete_flow_mwh
FROM system_load
ORDER BY date, period;
"""

load = pd.read_sql(query, connection)

connection.close()

load["date"] = pd.to_datetime(load["date"])

print("Raw shape:", load.shape)
display(load.head())

Raw shape: (33475, 4)


/var/folders/hl/r1nrlw8d6g54vkvsmknwfw7h0000gn/T/ipykernel_50403/630431578.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  load = pd.read_sql(query, connection)


,date,period,net_load_mwh,crete_flow_mwh
0,2023-01-01,1,4190.0,-28.0
1,2023-01-01,2,3961.0,-17.0
2,2023-01-01,3,3794.0,-3.0
3,2023-01-01,4,3585.0,-1.0
4,2023-01-01,5,3449.0,9.0


In [24]:
MIN_VALID_LOAD_MWH = 500

load_ml = (
    load[
        load["period"].between(1, 24)
    ]
    .copy()
    .reset_index(drop=True)
)

load_ml.loc[
    load_ml["net_load_mwh"] < MIN_VALID_LOAD_MWH,
    "net_load_mwh"
] = np.nan

print("Rows:", len(load_ml))
print(
    "Invalid load values:",
    load_ml["net_load_mwh"].isna().sum()
)

Rows: 32136
Invalid load values: 31


In [25]:
load_ml["timestamp"] = (
    load_ml["date"]
    + pd.to_timedelta(
        load_ml["period"] - 1,
        unit="h"
    )
)

load_ml = (
    load_ml
    .sort_values("timestamp")
    .reset_index(drop=True)
)

display(
    load_ml[
        [
            "timestamp",
            "date",
            "period",
            "net_load_mwh"
        ]
    ].head(30)
)

,timestamp,date,period,net_load_mwh
0,2023-01-01 00:00:00,2023-01-01,1,4190.0
1,2023-01-01 01:00:00,2023-01-01,2,3961.0
2,2023-01-01 02:00:00,2023-01-01,3,3794.0
3,2023-01-01 03:00:00,2023-01-01,4,3585.0
4,2023-01-01 04:00:00,2023-01-01,5,3449.0
5,2023-01-01 05:00:00,2023-01-01,6,3449.0
6,2023-01-01 06:00:00,2023-01-01,7,3513.0
7,2023-01-01 07:00:00,2023-01-01,8,3621.0
8,2023-01-01 08:00:00,2023-01-01,9,3473.0
9,2023-01-01 09:00:00,2023-01-01,10,3059.0


In [26]:
load_ml["hour"] = load_ml["timestamp"].dt.hour
load_ml["day_of_week"] = load_ml["timestamp"].dt.dayofweek
load_ml["month"] = load_ml["timestamp"].dt.month
load_ml["day_of_year"] = load_ml["timestamp"].dt.dayofyear

load_ml["is_weekend"] = (
    load_ml["day_of_week"] >= 5
).astype(int)

display(
    load_ml[
        [
            "timestamp",
            "hour",
            "day_of_week",
            "month",
            "day_of_year",
            "is_weekend"
        ]
    ].head()
)

,timestamp,hour,day_of_week,month,day_of_year,is_weekend
0,2023-01-01 00:00:00,0,6,1,1,1
1,2023-01-01 01:00:00,1,6,1,1,1
2,2023-01-01 02:00:00,2,6,1,1,1
3,2023-01-01 03:00:00,3,6,1,1,1
4,2023-01-01 04:00:00,4,6,1,1,1


In [27]:
load_ml["hour_sin"] = np.sin(
    2 * np.pi * load_ml["hour"] / 24
)

load_ml["hour_cos"] = np.cos(
    2 * np.pi * load_ml["hour"] / 24
)

load_ml["dow_sin"] = np.sin(
    2 * np.pi * load_ml["day_of_week"] / 7
)

load_ml["dow_cos"] = np.cos(
    2 * np.pi * load_ml["day_of_week"] / 7
)

load_ml["doy_sin"] = np.sin(
    2 * np.pi * load_ml["day_of_year"] / 365.25
)

load_ml["doy_cos"] = np.cos(
    2 * np.pi * load_ml["day_of_year"] / 365.25
)

display(
    load_ml[
        [
            "timestamp",
            "hour_sin",
            "hour_cos",
            "dow_sin",
            "dow_cos",
            "doy_sin",
            "doy_cos"
        ]
    ].head()
)

,timestamp,hour_sin,hour_cos,dow_sin,dow_cos,doy_sin,doy_cos
0,2023-01-01 00:00:00,0.000000,1.000000,-0.781831,0.62349,0.017202,0.999852
1,2023-01-01 01:00:00,0.258819,0.965926,-0.781831,0.62349,0.017202,0.999852
2,2023-01-01 02:00:00,0.500000,0.866025,-0.781831,0.62349,0.017202,0.999852
3,2023-01-01 03:00:00,0.707107,0.707107,-0.781831,0.62349,0.017202,0.999852
4,2023-01-01 04:00:00,0.866025,0.500000,-0.781831,0.62349,0.017202,0.999852


In [28]:
load_ml["load_lag_24"] = (
    load_ml["net_load_mwh"].shift(24)
)

load_ml["load_lag_48"] = (
    load_ml["net_load_mwh"].shift(48)
)

load_ml["load_lag_168"] = (
    load_ml["net_load_mwh"].shift(168)
)

display(
    load_ml[
        [
            "timestamp",
            "net_load_mwh",
            "load_lag_24",
            "load_lag_48",
            "load_lag_168"
        ]
    ].head(30)
)

,timestamp,net_load_mwh,load_lag_24,load_lag_48,load_lag_168
0,2023-01-01 00:00:00,4190.0,NaN,NaN,NaN
1,2023-01-01 01:00:00,3961.0,NaN,NaN,NaN
2,2023-01-01 02:00:00,3794.0,NaN,NaN,NaN
3,2023-01-01 03:00:00,3585.0,NaN,NaN,NaN
4,2023-01-01 04:00:00,3449.0,NaN,NaN,NaN
5,2023-01-01 05:00:00,3449.0,NaN,NaN,NaN
6,2023-01-01 06:00:00,3513.0,NaN,NaN,NaN
7,2023-01-01 07:00:00,3621.0,NaN,NaN,NaN
8,2023-01-01 08:00:00,3473.0,NaN,NaN,NaN
9,2023-01-01 09:00:00,3059.0,NaN,NaN,NaN


In [29]:
load_ml["load_lag_1"] = (
    load_ml["net_load_mwh"].shift(1)
)

load_ml["load_lag_24"] = (
    load_ml["net_load_mwh"].shift(24)
)

load_ml["load_lag_48"] = (
    load_ml["net_load_mwh"].shift(48)
)

load_ml["load_lag_168"] = (
    load_ml["net_load_mwh"].shift(168)
)

In [30]:
past_load = load_ml["net_load_mwh"].shift(1)

load_ml["rolling_mean_24"] = (
    past_load
    .rolling(24)
    .mean()
)

load_ml["rolling_std_24"] = (
    past_load
    .rolling(24)
    .std()
)

load_ml["rolling_mean_168"] = (
    past_load
    .rolling(168)
    .mean()
)

In [31]:
history_24h_ago = (
    load_ml["net_load_mwh"].shift(24)
)

load_ml["rolling_mean_24"] = (
    history_24h_ago
    .rolling(24)
    .mean()
)

load_ml["rolling_std_24"] = (
    history_24h_ago
    .rolling(24)
    .std()
)

load_ml["rolling_mean_168"] = (
    history_24h_ago
    .rolling(168)
    .mean()
)

In [32]:
load_ml["target"] = load_ml["net_load_mwh"]

In [33]:
feature_columns = [
    "hour",
    "day_of_week",
    "month",
    "day_of_year",
    "is_weekend",

    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "doy_sin",
    "doy_cos",

    "load_lag_24",
    "load_lag_48",
    "load_lag_168",

    "rolling_mean_24",
    "rolling_std_24",
    "rolling_mean_168",
]

print("Number of features:", len(feature_columns))

Number of features: 17


In [34]:
model_data = (
    load_ml[
        [
            "timestamp",
            "target",
            *feature_columns
        ]
    ]
    .dropna()
    .reset_index(drop=True)
)

print("Final shape:", model_data.shape)
print(
    "Missing values:",
    model_data.isna().sum().sum()
)

display(model_data.head())

Final shape: (30309, 19)
Missing values: 0


,timestamp,target,hour,day_of_week,month,day_of_year,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos,doy_sin,doy_cos,load_lag_24,load_lag_48,load_lag_168,rolling_mean_24,rolling_std_24,rolling_mean_168
0,2023-01-08 23:00:00,4847.0,23,6,1,8,1,-0.258819,0.965926,-0.781831,0.62349,0.137185,0.990545,4915.0,4698.0,4288.0,4158.666667,1113.275413,4273.083333
1,2023-01-09 00:00:00,4352.0,0,0,1,9,0,0.000000,1.000000,0.000000,1.00000,0.154204,0.988039,4526.0,4278.0,3936.0,4169.000000,1115.579787,4275.083333
2,2023-01-09 01:00:00,3961.0,1,0,1,9,0,0.258819,0.965926,0.000000,1.00000,0.154204,0.988039,4157.0,3951.0,3630.0,4177.583333,1114.621633,4276.250000
3,2023-01-09 02:00:00,3883.0,2,0,1,9,0,0.500000,0.866025,0.000000,1.00000,0.154204,0.988039,4026.0,3874.0,3560.0,4183.916667,1113.252654,4277.630952
4,2023-01-09 03:00:00,3779.0,3,0,1,9,0,0.707107,0.707107,0.000000,1.00000,0.154204,0.988039,3852.0,3732.0,3469.0,4188.916667,1111.402636,4279.220238


In [35]:
print("Start:", model_data["timestamp"].min())
print("End:", model_data["timestamp"].max())

print(
    "Duplicate timestamps:",
    model_data["timestamp"].duplicated().sum()
)

print(
    "Minimum target:",
    model_data["target"].min()
)

print(
    "Maximum target:",
    model_data["target"].max()
)

Start: 2023-01-08 23:00:00
End: 2026-08-31 23:00:00
Duplicate timestamps: 0
Minimum target: 500.0
Maximum target: 9141.0


In [36]:
output_path = "../data/processed/ml_load_features.csv"

model_data.to_csv(
    output_path,
    index=False
)

print(f"Saved: {output_path}")

Saved: ../data/processed/ml_load_features.csv


In [39]:
%whos DataFrame

Variable         Type         Data/Info
---------------------------------------
load             DataFrame    Shape: (33475, 4)
load_ml          DataFrame    Shape: (32136, 24)
model_data       DataFrame    Shape: (30309, 19)
smallest_loads   DataFrame    Shape: (20, 4)
suspicious       DataFrame    Shape: (92, 4)


In [40]:
model_data.nsmallest(20, "target")[
    ["timestamp", "target", "load_lag_24", "load_lag_48", "load_lag_168"]
]


,timestamp,target,load_lag_24,load_lag_48,load_lag_168
1838,2023-03-26 13:00:00,500.0,901.0,1825.0,1492.0
18815,2025-04-18 11:00:00,513.0,1649.0,1992.0,1684.0
18814,2025-04-18 10:00:00,560.0,2106.0,2682.0,2286.0
11023,2024-05-03 13:00:00,572.0,1847.0,2443.0,1359.0
11024,2024-05-03 15:00:00,586.0,2106.0,2289.0,1492.0
10259,2024-03-18 11:00:00,603.0,2202.0,1701.0,2795.0
18817,2025-04-18 13:00:00,635.0,1331.0,2118.0,1768.0
18818,2025-04-18 14:00:00,644.0,1795.0,2067.0,1634.0
1839,2023-03-26 14:00:00,655.0,1018.0,1847.0,1492.0
19728,2025-05-26 12:00:00,673.0,3403.0,1869.0,939.0


In [41]:
model_data[model_data["target"] <= 1000][
    ["timestamp", "target"]
]

,timestamp,target
1812,2023-03-25 11:00:00,792.0
1813,2023-03-25 12:00:00,716.0
1814,2023-03-25 13:00:00,901.0
1836,2023-03-26 11:00:00,862.0
1837,2023-03-26 12:00:00,728.0
1838,2023-03-26 13:00:00,500.0
1839,2023-03-26 14:00:00,655.0
2679,2023-05-07 15:00:00,997.0
6374,2023-10-08 14:00:00,965.0
10259,2024-03-18 11:00:00,603.0


In [42]:

from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

model_data.to_csv(
    "../data/processed/model_data.csv",
    index=False
)

print("DONE")

DONE
